In [191]:
import pandas as pd 

data = pd.read_csv("diabetes2.csv")

data

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1
...,...,...,...,...,...,...,...,...,...
763,10,101,76,48,180,32.9,0.171,63,0
764,2,122,70,27,0,36.8,0.340,27,0
765,5,121,72,23,112,26.2,0.245,30,0
766,1,126,60,0,0,30.1,0.349,47,1


In [192]:
data.info(memory_usage='deep')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 non-null    int64  
 2   BloodPressure             768 non-null    int64  
 3   SkinThickness             768 non-null    int64  
 4   Insulin                   768 non-null    int64  
 5   BMI                       768 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB


In [193]:
# some impossible, missing values, like 0 for glucose, skinthickness, etc. should be imputed later with median value
data.query("Glucose == 0 ") 

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
75,1,0,48,20,0,24.7,0.140,22,0
182,1,0,74,20,23,27.7,0.299,21,0
342,1,0,68,35,0,32.0,0.389,22,0
349,5,0,80,32,0,41.0,0.346,37,1
502,6,0,68,41,0,39.0,0.727,41,1


In [194]:
X = data.iloc[:,:8]
y = data.iloc[:, 8]

In [195]:
data.Outcome.value_counts()

Outcome
0    500
1    268
Name: count, dtype: int64

In [196]:
data.Pregnancies.value_counts().sort_index()

Pregnancies
0     111
1     135
2     103
3      75
4      68
5      57
6      50
7      45
8      38
9      28
10     24
11     11
12      9
13     10
14      2
15      1
17      1
Name: count, dtype: int64

In [197]:
from sklearn.feature_selection import mutual_info_classif

mi = mutual_info_classif(X, y, discrete_features=False)

mi_scores = pd.Series(mi, index=X.columns).sort_values(ascending=False)

mi_scores

Glucose                     0.111714
Age                         0.081333
BMI                         0.061942
BloodPressure               0.026769
Insulin                     0.021654
DiabetesPedigreeFunction    0.010243
SkinThickness               0.000200
Pregnancies                 0.000000
dtype: float64

In [198]:
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

num_cols = X.iloc[:,1:8].columns
preg = ['Pregnancies']

num_pipeline = Pipeline([
    ("imputer", SimpleImputer(missing_values=0, strategy='median')),
    ("scaler", StandardScaler()),

])

pregnancy_pipine = Pipeline([
    ("scaler", StandardScaler()),

])

preprocess = ColumnTransformer([
    ("num", num_pipeline, num_cols),
    ("preg", pregnancy_pipine, preg)
])

pipe = Pipeline([
    ("preprocess", preprocess),
    ("model", LogisticRegression(max_iter=2000))
])


X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.3, random_state=42, stratify=y)

In [199]:
from sklearn.model_selection import cross_val_score

cv = cross_val_score(pipe, X_train, y_train, cv=5, scoring="roc_auc").mean()

cv

np.float64(0.8358077626498679)

<h4> Hyperparameter Tuning </h4>


In [200]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "model__C": [0.001, 0.01, 0.1, 1, 10, 100],
    "model__class_weight": [None, "balanced"]
}

gs = GridSearchCV(
    pipe,
    param_grid,
    cv=5,
    scoring="roc_auc"
)

gs.fit(X_train, y_train)

gs.best_score_, gs.best_params_


(np.float64(0.8358077626498679), {'model__C': 1, 'model__class_weight': None})

In [201]:
pipe = gs.best_estimator_

In [202]:
from sklearn.metrics import roc_auc_score


pipe.fit(X_train, y_train)

y_prob = pipe.predict_proba(X_test)[:, 1]
y_pred = pipe.predict(X_test)
roc_auc = roc_auc_score(y_test, y_prob)
roc_auc



np.float64(0.8362962962962962)

In [203]:
X_test['True_Label'] = y_test
X_test['Prediction'] = y_pred
X_test['Diabetes_Probability'] = y_prob
X_test

X_test.loc[X_test.Diabetes_Probability.idxmax()]


Pregnancies                   0.000000
Glucose                     180.000000
BloodPressure                78.000000
SkinThickness                63.000000
Insulin                      14.000000
BMI                          59.400000
DiabetesPedigreeFunction      2.420000
Age                          25.000000
True_Label                    1.000000
Prediction                    1.000000
Diabetes_Probability          0.989654
Name: 445, dtype: float64

In [204]:
X_test.loc[X_test.Diabetes_Probability.idxmin()]

Pregnancies                  0.000000
Glucose                     73.000000
BloodPressure                0.000000
SkinThickness                0.000000
Insulin                      0.000000
BMI                         21.100000
DiabetesPedigreeFunction     0.342000
Age                         25.000000
True_Label                   0.000000
Prediction                   0.000000
Diabetes_Probability         0.009567
Name: 589, dtype: float64

In [205]:
len(X_test.query("True_Label != Prediction"))

59

# Model 1 Conclusion

With no feature engineering, the ROC_AUC score is 0.8362962962962962. 

This is with hyperparameters {'model__C': 1, 'model__class_weight': None} .


# Model 2 with Different Features

In [206]:
data.Insulin.describe()

count    768.000000
mean      79.799479
std      115.244002
min        0.000000
25%        0.000000
50%       30.500000
75%      127.250000
max      846.000000
Name: Insulin, dtype: float64

In [207]:
data.DiabetesPedigreeFunction.describe()

count    768.000000
mean       0.471876
std        0.331329
min        0.078000
25%        0.243750
50%        0.372500
75%        0.626250
max        2.420000
Name: DiabetesPedigreeFunction, dtype: float64

Insulin and DiabetesPedigreeFunction are not evenly distributed at all. 

75% of insulin values are below 127, the rest go up to 846. 

The values are very skewed, I can calm them down using log.

In [208]:
import numpy as np

X['glucose_bmi'] = X.Glucose * X.BMI
X["age_preg_ratio"] = X["Pregnancies"] / (X["Age"] + 1)

X["log_insulin"] = np.log1p(X["Insulin"])
X["log_dpf"] = np.log1p(X["DiabetesPedigreeFunction"])

X = X.drop(columns=['Insulin', 'Age', 'Pregnancies', 'BMI', 'DiabetesPedigreeFunction', 'Glucose'])
X


,BloodPressure,SkinThickness,glucose_bmi,age_preg_ratio,log_insulin,log_dpf
0,72,35,4972.8,0.117647,0.000000,0.486738
1,66,29,2261.0,0.031250,0.000000,0.300845
2,64,0,4263.9,0.242424,0.000000,0.514021
3,66,23,2500.9,0.045455,4.553877,0.154436
4,40,35,5904.7,0.000000,5.129899,1.190279
...,...,...,...,...,...,...
763,76,48,3322.9,0.156250,5.198497,0.157858
764,70,27,4489.6,0.071429,0.000000,0.292670
765,72,23,3170.2,0.161290,4.727388,0.219136
766,60,0,3792.6,0.020833,0.000000,0.299364


In [209]:
mi = pd.Series(mutual_info_classif(X, y, discrete_features=False), index=X.columns).sort_values(ascending=False)
mi

glucose_bmi       0.130934
age_preg_ratio    0.064283
log_insulin       0.037962
SkinThickness     0.021119
log_dpf           0.016913
BloodPressure     0.002919
dtype: float64

In [210]:
preprocess = ColumnTransformer([
    ('num', StandardScaler(), X.columns)
])


pipe = Pipeline([
    ('preprocess', preprocess),
    ('model', LogisticRegression(max_iter=1500, penalty='l2', solver='lbfgs', C=1.0))
])


X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, stratify=y, test_size=0.3)

In [211]:
param_grid = {
    "model__C": [0.001, 0.01, 0.1, 1, 10],
    "model__solver": ["lbfgs"],
    "model__penalty": ["l2"]
}


gs = GridSearchCV(
    pipe,
    param_grid,
    cv=5,
    scoring="roc_auc"
)

gs.fit(X_train, y_train)

gs.best_score_, gs.best_params_

(np.float64(0.8244665718349931),
 {'model__C': 0.1, 'model__penalty': 'l2', 'model__solver': 'lbfgs'})

In [212]:
pipe = gs.best_estimator_

pipe.fit(X_train, y_train)

y_prob = pipe.predict_proba(X_test)[:, 1]
y_pred = pipe.predict(X_test)
roc_auc = roc_auc_score(y_test, y_prob)
roc_auc

np.float64(0.8299588477366255)

In [213]:
X_test['True_Label'] = y_test
X_test['Prediction'] = y_pred
X_test['Diabetes_Probability'] = y_prob
X_test


,BloodPressure,SkinThickness,glucose_bmi,age_preg_ratio,log_insulin,log_dpf,True_Label,Prediction,Diabetes_Probability
730,78,23,3692.0,0.085714,4.382027,0.279902,1,0,0.218073
198,64,44,3793.2,0.148148,4.605170,0.644482,1,0,0.377814
24,94,33,5233.8,0.211538,4.990433,0.226338,1,1,0.583079
417,82,32,5544.0,0.105263,0.000000,0.440832,1,1,0.629463
387,100,36,4546.5,0.173913,0.000000,0.214305,1,0,0.428547
...,...,...,...,...,...,...,...,...,...
94,82,18,3507.4,0.090909,4.174387,0.565882,0,0,0.256783
437,75,0,4395.3,0.172414,0.000000,0.360468,0,1,0.508745
86,72,54,3879.6,0.282609,0.000000,0.163818,0,0,0.433309
221,90,0,4992.8,0.029851,0.000000,0.590561,1,0,0.498783


In [214]:
len(X_test.query("True_Label != Prediction"))

60

# Conclusion

Model 1 where I kept all features, scaled them, and imputed impossible values (such as zero glucose, insulin, skinthickness) achieved a ROC_AUC score of 0.83.

Trying to improve the model, I engineered new features (Glucose * BMI, Pregnancies / Age) and applied log to DiabetesPedigreeFunction & Insulin, but this decreased the ROC_AUC to 0.82.

